# บทเรียนที่ 13 - ความทรงจำของตัวแทน


## การตั้งค่า

สมุดโน้ตนี้สาธิตวิธีการสร้างตัวแทนจองการเดินทางที่มี **หน่วยความจำถาวร** โดยใช้ **Microsoft Agent Framework** (MAF)

คุณจะได้เรียนรู้ว่าหน่วยความจำประเภทต่างๆ ของตัวแทน — หน่วยความจำทำงาน หน่วยความจำระยะสั้น และหน่วยความจำระยะยาว — มีผลอย่างไรต่อการที่ตัวแทนเก็บรักษาและใช้ข้อมูลในหลายบทสนทนา

**ข้อกำหนดเบื้องต้น:**
- โครงการ Azure AI Foundry ที่มีโมเดลแชทที่ปรับใช้แล้ว (เช่น `gpt-4o-mini`)
- เข้าสู่ระบบด้วย Azure CLI — รันคำสั่ง `az login` ในเทอร์มินัลของคุณ
- `AZURE_AI_PROJECT_ENDPOINT` — จุดสิ้นสุดของโครงการ Azure AI Foundry ของคุณ
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ชื่อของโมเดลที่คุณปรับใช้แล้ว


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ประเภทของความจำของเอเจนต์

เอเจนต์ AI สามารถใช้ประโยชน์จากความจำประเภทต่างๆ ซึ่งแต่ละประเภทมีวัตถุประสงค์เฉพาะเจาะจง:

### ความจำระยะสั้น (Working Memory)
เส้นบทสนทนาเอง — ข้อความที่แลกเปลี่ยนกันในเซสชันเดียว เอเจนต์สามารถอ้างอิงกลับไปยังข้อความก่อนหน้าในเส้นบทสนทนาเดียวกันเพื่อรักษาความสอดคล้อง ใน MAF นี้ถูกสร้างขึ้นผ่าน **`agent.create_session()`** ซึ่งจะส่งกลับ `AgentSession`

### ความจำระยะสั้น (Short-Term Memory)
ข้อมูลที่คงอยู่ในระหว่างเวลาของงานหรือเซสชันแต่ไม่ได้ถูกเก็บไว้อย่างถาวร ตัวอย่างเช่น เอเจนต์อาจสะสมข้อเท็จจริงระหว่างการวางแผนหลายรอบการสนทนาและใช้เพื่อสร้างแผนการเดินทางขั้นสุดท้าย

### ความจำระยะยาว (Long-Term Memory)
ความชอบและข้อเท็จจริงที่คงอยู่ **ข้ามเซสชัน** ผู้ใช้ที่กลับมาไม่ควรต้องพูดซ้ำข้อจำกัดทางอาหารหรือสไตล์การท่องเที่ยวของพวกเขา ความจำระยะยาวโดยทั่วไปจะถูกสนับสนุนโดยแหล่งเก็บข้อมูลภายนอก — ฐานข้อมูล ไฟล์ หรือดัชนีเวกเตอร์ — และถูกแสดงผลให้เอเจนต์ผ่านทางเครื่องมือ


## Working Memory with Sessions

รูปแบบหน่วยความจำที่ง่ายที่สุดคือเซสชันการสนทนา เมื่อคุณส่งผ่านเซสชันเดียวกัน (สร้างขึ้นผ่าน `agent.create_session()`) ไปยังการเรียก `agent.run()` ติดต่อกัน เอเย่นต์จะเห็นประวัติเต็มของการสนทนานั้นและสามารถเรียกคืนรายละเอียดก่อนหน้าได้

มาสร้างเอเย่นต์ท่องเที่ยวและสาธิตหน่วยความจำที่ใช้งานกันเถอะ


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ตัวแทนจำงบประมาณได้อย่างถูกต้องเพราะข้อความทั้งสองแชร์เซสชันเดียวกัน นี่คือ **ความจำทำงาน** — ซึ่งมีอยู่เพียงตลอดช่วงอายุของเซสชันเท่านั้น

### จะเกิดอะไรขึ้นกับเธรดใหม่?

ถ้าเราสร้างเซสชัน **ใหม่** ตัวแทนจะไม่มีความทรงจำของการสนทนาก่อนหน้า:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## รูปแบบหน่วยความจำระยะยาว

เพื่อจดจำความชอบของผู้ใช้งาน **ข้ามเซสชัน** เราจำเป็นต้องมีที่เก็บข้อมูลถาวรที่อยู่นอกเหนือจากเส้นสนทนา ตัวแทนจะเข้าถึงที่เก็บนี้ผ่านทาง **เครื่องมือ** — ฟังก์ชันที่มันสามารถเรียกใช้เพื่อบันทึกและดึงข้อมูล

ด้านล่างนี้เราจะสร้างที่เก็บความชอบแบบอยู่ในหน่วยความจำอย่างง่าย (ในสภาพแวดล้อมจริงคุณจะต้องใช้ฐานข้อมูลหรือดัชนีเวกเตอร์มาสนับสนุน) และเผยให้เป็นเครื่องมือที่ตัวแทนสามารถใช้งานได้

### สถาปัตยกรรม
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — ผู้ใช้ครั้งแรกจองทริปวันครบรอบ

ซาร่าห์เยี่ยมชมเป็นครั้งแรก ตัวแทนควรจัดเก็บความชอบของเธอผ่านเครื่องมือและใช้ข้อมูลนั้นเพื่อแนะนำโรงแรมให้เธอได้


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### สถานการณ์ที่ 2 — ซาร่าห์กลับมาอีกหลายสัปดาห์ต่อมา

ซาร่าห์เริ่ม **เธรดใหม่เอี่ยม** (จำลองเซสชันใหม่) หน่วยความจำใช้งานว่างเปล่า แต่ที่เก็บข้อมูลความชอบระยะยาวยังคงมีข้อมูลของเธออยู่ ตัวแทนควรดึงข้อมูลนั้นมาใช้และใช้เพื่อปรับแต่งคำแนะนำให้เหมาะกับเธอ


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## สรุป

ในบทเรียนนี้ คุณได้เรียนรู้เกี่ยวกับหน่วยความจำตัวแทนสามประเภทและวิธีการใช้งานกับ Microsoft Agent Framework:

| ประเภทหน่วยความจำ | กลไก MAF | อายุการใช้งาน |
|---|---|---|
| **ทำงาน** | `agent.create_session()` | การสนทนาเพียงครั้งเดียว |
| **ระยะสั้น** | บริบทสะสมภายในเธรด | งาน / เซสชันเดียว |
| **ระยะยาว** | เก็บข้อมูลภายนอกที่เข้าถึงผ่านฟังก์ชัน `@tool` | ข้ามเซสชัน |

### ประเด็นสำคัญ
1. **`agent.create_session()`** ให้หน่วยความจำทำงาน — ตัวแทนจะเห็นประวัติการสนทนาเต็มภายในเซสชัน
2. **เซสชันใหม่จะสูญเสียบริบท** — หากไม่มีหน่วยความจำระยะยาว ตัวแทนจะไม่สามารถจำการสนทนาในอดีตได้
3. **ฟังก์ชัน `@tool`** ช่วยเชื่อมโยงช่องว่าง — ให้ตัวแทนสามารถบันทึกและดึงข้อมูลจากพื้นที่เก็บข้อมูลถาวร
4. **การปรับแต่งเฉพาะบุคคลจะดีขึ้นตามเวลา** — ยิ่งมีการเก็บข้อมูลความชอบมากเท่าไหร่ การแนะนำของตัวแทนก็จะยิ่งดีขึ้น

### การใช้งานในโลกจริง
- **บริการลูกค้า**: จดจำประวัติและความชอบของลูกค้า
- **ผู้ช่วยส่วนตัว**: รักษาบริบทข้ามวันหรือสัปดาห์
- **สุขภาพ**: ติดตามข้อมูลและความชอบของผู้ป่วย
- **อีคอมเมิร์ซ**: การช็อปปิ้งแบบเฉพาะบุคคลตามประวัติ

### ขั้นตอนต่อไป
- แทนที่ dict ในหน่วยความจำด้วยฐานข้อมูลหรือ vector store (เช่น Azure AI Search)
- เพิ่มการหมดอายุของหน่วยความจำสำหรับข้อมูลที่มีความสำคัญตามเวลา
- สร้างระบบตัวแทนหลายตัวที่มีหน่วยความจำร่วมกัน
- สำรวจสมุดบันทึก Cognee สำหรับหน่วยความจำที่สนับสนุนด้วยความรู้กราฟ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้ว่าเราจะพยายามให้ความถูกต้องสูงสุด โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้องได้ เอกสารต้นฉบับในภาษาต้นทางควรถือเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่มีความสำคัญ ควรใช้บริการแปลโดยนักแปลมืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
